# Descarga y preparación de dataset para nanotoxicidad

Esta notebook descarga y prepara una base pública de nanotoxicidad para iniciar la implementación del proyecto integrador de la Unidad 6.

Fuente principal recomendada:
- Zenodo: Structured Nanotoxicity Datasets with Physicochemical and Toxicological Attributes of Metal Oxide Nanoparticles
- DOI: https://doi.org/10.5281/zenodo.15385143

Conjunto objetivo para arrancar:
- `HaHa-Manual.csv` por su tamaño y curación manual
- `HA3B.csv` como subconjunto pequeño de validación

La notebook deja el flujo listo para:
- cargar CSV públicos
- consolidar columnas
- inspeccionar variables
- detectar faltantes
- construir una base utilizable en U6_03_IMPLEMENTACION_PROYECTO.ipynb

## 1. Qué vamos a descargar

El record de Zenodo expone tres CSV relevantes:
- `HaHa-Auto.csv`
- `HaHa-Manual.csv`
- `HA3B.csv`

**Recomendación práctica**
- usar `HaHa-Manual.csv` como base principal
- usar `HA3B.csv` como conjunto auxiliar o de validación

**Por qué**
- `HaHa-Manual` tiene curación manual y más filas que `HA3B`
- `HA3B` es pequeño y útil para validación rápida
- ambos traen atributos fisicoquímicos y endpoints de toxicidad

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

sns.set_theme(style='whitegrid')

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw' / 'zenodo_nanotoxicity'
PROCESSED_DIR = DATA_DIR / 'processed'
FIGURES_DIR = ROOT / 'figuras'
for folder in [RAW_DIR, PROCESSED_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Carpetas listas:')
print('-', RAW_DIR)
print('-', PROCESSED_DIR)
print('-', FIGURES_DIR)

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 2. Descarga de archivos CSV

Se intenta descargar el CSV directamente desde Zenodo.

Si el enlace directo cambiara, también puedes descargar manualmente los archivos desde la página del record y colocarlos en la carpeta `data/raw/zenodo_nanotoxicity/`.

En esta notebook se usa un patrón de descarga simple y reproducible.

In [ ]:
ZENODO_BASE = 'https://zenodo.org/records/15385143/files'
FILES = {
    'HaHa-Manual.csv': f'{ZENODO_BASE}/HaHa-Manual.csv?download=1',
    'HA3B.csv': f'{ZENODO_BASE}/HA3B.csv?download=1',
    'HaHa-Auto.csv': f'{ZENODO_BASE}/HaHa-Auto.csv?download=1',
}

def download_file(url: str, out_path: Path) -> bool:
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        out_path.write_bytes(response.content)
        return True
    except Exception as exc:
        print(f'No se pudo descargar {out_path.name}: {exc}')
        return False

downloaded = {}
for filename, url in FILES.items():
    out_path = RAW_DIR / filename
    if out_path.exists():
        downloaded[filename] = 'ya_existia'
        continue
    ok = download_file(url, out_path)
    downloaded[filename] = 'descargado' if ok else 'fallo'

print(json.dumps(downloaded, ensure_ascii=False, indent=2))

## 3. Carga de los datasets

Se cargan los CSV disponibles y se selecciona la base principal para comenzar.

La lógica prioriza `HaHa-Manual.csv`, luego `HA3B.csv`, y por último `HaHa-Auto.csv`.

In [ ]:
def load_csv_if_exists(path: Path):
    if path.exists():
        return pd.read_csv(path)
    return None

datasets = {}
for filename in ['HaHa-Manual.csv', 'HA3B.csv', 'HaHa-Auto.csv']:
    path = RAW_DIR / filename
    df = load_csv_if_exists(path)
    if df is not None:
        datasets[filename] = df
        print(f'{filename}: {df.shape[0]} filas x {df.shape[1]} columnas')

if not datasets:
    raise FileNotFoundError('No se pudo cargar ningún CSV de Zenodo.')

priority = ['HaHa-Manual.csv', 'HA3B.csv', 'HaHa-Auto.csv']
for key in priority:
    if key in datasets:
        df = datasets[key].copy()
        source_name = key
        break

print(f'Base principal seleccionada: {source_name}')
df.head()

## 4. Inspección de variables

Aquí identificamos automáticamente:
- variables numéricas
- variables categóricas
- valores faltantes
- duplicados

Además se intenta detectar posibles columnas candidatas para el target de toxicidad.

In [ ]:
df.columns = [c.strip().lower() for c in df.columns]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print('Columnas numéricas:')
print(numeric_cols)
print('\nColumnas categóricas:')
print(categorical_cols)
print('\nValores faltantes por columna:')
print(df.isna().sum().sort_values(ascending=False))
print('\nDuplicados:', df.duplicated().sum())

display(df[numeric_cols].describe().T if numeric_cols else pd.DataFrame())

for col in categorical_cols:
    print(f'\nFrecuencias de {col}:')
    print(df[col].value_counts(dropna=False).head(10))

## 5. Variables esperadas en este tipo de dataset

En este tipo de base pública suelen aparecer variables como:

- tamaño de núcleo o tamaño hidrodinámico
- potencial zeta / carga superficial
- composición química
- área superficial
- band gap / descriptores cuánticos
- tipo de célula o bioensayo
- dosis o concentración
- tiempo de exposición
- endpoint de toxicidad o viabilidad

Si las columnas exactas cambian, la notebook sigue siendo útil porque la inspección es automática y el pipeline se adapta al esquema real.

## 6. Identificación del target

El objetivo ideal para el proyecto es una variable asociada a toxicidad o viabilidad.

Candidatos típicos:
- `toxicity`
- `toxic`
- `viability`
- `cell_viability`
- `endpoint`
- `response`
- `effect`

Si no existe una columna explícita, se puede construir una etiqueta binaria a partir del endpoint reportado y un umbral justificado.

En esta notebook se deja una función de detección de columnas candidatas.

In [ ]:
target_candidates = [
    'toxicity', 'toxic', 'toxicity_score', 'viability', 'cell_viability',
    'endpoint', 'response', 'effect', 'cytotoxicity', 'hazard'
]

found_targets = [c for c in df.columns if any(t in c for t in target_candidates)]
print('Candidatos a target detectados:')
print(found_targets if found_targets else 'No se detectó un target explícito con este criterio.')

target_col = found_targets[0] if found_targets else None
print('Target elegido:', target_col)

## 7. Construcción de una clasificación binaria provisional

Si el dataset trae una variable continua, se puede crear una versión binaria simple:

- `toxic`
- `non_toxic`

La regla exacta dependerá del tipo de target real que tenga el CSV.

Ejemplos:
- si el valor es viabilidad, menor viabilidad implica mayor toxicidad
- si el valor es IC50 o LC50, un umbral experimental define toxicidad
- si ya existe una etiqueta binaria, se respeta tal como viene

In [ ]:
def build_binary_target(frame: pd.DataFrame, target: str | None):
    if target is None:
        return None, None

    series = frame[target].copy()

    if series.dtype == 'object':
        normalized = series.astype(str).str.lower().str.strip()
        mapping = {
            'toxic': 'toxic',
            'non-toxic': 'non_toxic',
            'non toxic': 'non_toxic',
            'nontoxic': 'non_toxic',
            '0': 'non_toxic',
            '1': 'toxic',
        }
        binary = normalized.map(lambda x: mapping.get(x, x))
        return binary, 'direct'

    numeric = pd.to_numeric(series, errors='coerce')
    if numeric.dropna().empty:
        return None, None

    if 'viability' in target or 'survival' in target:
        threshold = numeric.median()
        binary = np.where(numeric <= threshold, 'toxic', 'non_toxic')
        return pd.Series(binary, index=frame.index), f'viability_median_threshold={threshold:.4f}'

    threshold = numeric.median()
    binary = np.where(numeric >= threshold, 'toxic', 'non_toxic')
    return pd.Series(binary, index=frame.index), f'numeric_median_threshold={threshold:.4f}'

df_model = df.copy()
binary_target, target_rule = build_binary_target(df_model, target_col)

if binary_target is not None:
    df_model['target_binary'] = binary_target
    print('Regla aplicada:', target_rule)
    print(df_model['target_binary'].value_counts(dropna=False))
else:
    print('No fue posible construir un target binario con el criterio automático actual.')

df_model.head()

## 8. Limpieza mínima y preparación del dataset

Se hace una limpieza inicial para dejar el archivo listo para entrenamiento:
- columnas en minúsculas
- eliminación de duplicados
- guardado en la carpeta `data/processed/`

Si el target binario quedó disponible, también se guarda una versión ya lista para modelado.

In [ ]:
df_clean = df_model.drop_duplicates().reset_index(drop=True)
clean_path = PROCESSED_DIR / f'{source_name.replace(.csv, "").lower()}_clean.csv'
df_clean.to_csv(clean_path, index=False)

print(f'Dataset limpio guardado en: {clean_path}')
print(f'Forma limpia: {df_clean.shape[0]} filas x {df_clean.shape[1]} columnas')
print('Faltantes por columna (top 20):')
print(df_clean.isna().sum().sort_values(ascending=False).head(20))

## 9. Preparación del pipeline para U6_03

Si ya existe `target_binary`, esta sección deja preparado el flujo de entrenamiento con separación train/test y preprocesamiento.

El pipeline es compatible con la estructura de la Unidad 6 y con el despliegue posterior en FastAPI.

In [ ]:
if 'target_binary' in df_clean.columns:
    target_col = 'target_binary'
    feature_cols = [c for c in df_clean.columns if c != target_col]

    X = df_clean[feature_cols].copy()
    y = df_clean[target_col].copy()

    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() > 1 else None
    )

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features),
        ]
    )

    print('Features numéricas:', numeric_features)
    print('Features categóricas:', categorical_features)
    print('Tamaño train:', X_train.shape, y_train.shape)
    print('Tamaño test:', X_test.shape, y_test.shape)
else:
    print('No se creó target_binary automáticamente; revisa la columna de toxicidad real.')

## 10. Exploración visual inicial

Estas gráficas sirven para documentar la estructura del dataset y comunicar hallazgos iniciales.

Si el dataset tiene variables adecuadas, se generan histogramas, conteos y una vista rápida del target.

In [ ]:
plot_cols = []
for candidate in ['core size', 'size', 'hydrodynamic size', 'zeta potential', 'surface charge', 'concentration', 'dose', 'time', 'exposure time']:
    matches = [c for c in df_clean.columns if candidate.replace(' ', '') in c.replace(' ', '')]
    plot_cols.extend(matches)
plot_cols = list(dict.fromkeys(plot_cols))[:4]

if plot_cols:
    fig, axes = plt.subplots(len(plot_cols), 1, figsize=(10, 4 * len(plot_cols)))
    if len(plot_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, plot_cols):
        sns.histplot(df_clean[col].dropna(), kde=True, ax=ax, color='steelblue')
        ax.set_title(f'Distribución de {col}')
    plt.tight_layout()
    plt.show()
else:
    print('No se detectaron columnas típicas para gráficas automáticas.')

if 'target_binary' in df_clean.columns:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df_clean, x='target_binary', palette='Set2')
    plt.title('Distribución del target binario')
    plt.tight_layout()
    plt.show()

## 11. Resumen para el reporte y la integración multiagente

**Dataset principal recomendado**
- `HaHa-Manual.csv`

**Uso recomendado**
- base inicial del flujo de nanotoxicidad
- fuente de features y labels para un primer baseline
- referencia para conectar con `U6_03_IMPLEMENTACION_PROYECTO.ipynb`

**Conexión con `toxicity_predictor.py`**
- usar como safety gate heurístico
- marcar candidatos de alto riesgo
- servir como validación rápida del sistema multiagente

**Conexión futura con agentes**
- Data Agent: carga y limpieza del CSV
- Model Agent: entrenamiento del baseline
- Safety Gate: validación rápida
- Evaluation Agent: métricas y reporte
- API Agent: despliegue con FastAPI